# Register Steering v2 — Robust Quantitative Evaluation

Repeats the register steering experiment with:
- **50 prompts** (vs 4 in v1) for statistical power
- **Lexical metric**: counts register-specific terms in the output
- **Shift score v2**: target-feature activation (same metric as v1, with larger n)
- **Bootstrap 95% CI** for both metrics

**Runtime:** ~25 min on T4 | **Output:** `exp_register_steering_v2_results.json`

In [ ]:
!pip install transformer_lens sae_lens -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "./data/checkpoints/"
import os
os.makedirs(SAVE_DIR, exist_ok=True)

## Setup: Model + SAE

In [ ]:
import torch
import numpy as np
import json
import os
import re
import time
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"
HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"

from sae_lens import SAE, HookedSAETransformer

print("Carregando Gemma 2 2B...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b", device=device, dtype=torch.float16,
)
tokenizer = model.tokenizer
print(f"Modelo: {model.cfg.model_name}")

print(f"Carregando SAE: {SAE_ID}...")
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=device)
print(f"SAE: {sae.cfg.d_sae} features")

## Generation + metrics functions

In [ ]:
def generate_with_steering(model, sae, tokenizer, prompt, feature_ids,
                          multipliers, max_new_tokens=100, temperature=0.7,
                          hook_name=HOOK_NAME):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    feature_ids_t = torch.tensor(feature_ids, device=device)
    multipliers_t = torch.tensor(multipliers, dtype=torch.float16, device=device)

    def steering_hook(value, hook):
        with torch.no_grad():
            sae_input = value
            sae_acts = sae.encode(sae_input)
            modified_acts = sae_acts.clone()
            for fid, mult in zip(feature_ids_t, multipliers_t):
                modified_acts[:, :, fid] = modified_acts[:, :, fid] * mult
            reconstructed = sae.decode(modified_acts)
            error = sae_input - sae.decode(sae_acts)
            return reconstructed + error

    generated = input_ids.clone()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model.run_with_hooks(
                generated, fwd_hooks=[(hook_name, steering_hook)],
            )
            next_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            if next_token.item() == tokenizer.eos_token_id:
                break
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)

    return tokenizer.decode(generated[0], skip_special_tokens=True)


def generate_baseline(model, tokenizer, prompt, max_new_tokens=100,
                      temperature=0.7):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(generated)
            next_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            if next_token.item() == tokenizer.eos_token_id:
                break
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)
    return tokenizer.decode(generated[0], skip_special_tokens=True)


def measure_feature_activations(model, sae, tokenizer, text, feature_ids,
                                hook_name=HOOK_NAME):
    input_ids = tokenizer.encode(text, return_tensors="pt").to(device)
    with torch.no_grad():
        _, cache = model.run_with_cache(input_ids, names_filter=lambda n: n == hook_name)
        acts = cache[hook_name]
        feat_acts = sae.encode(acts)
    result = {}
    for fid in feature_ids:
        vals = feat_acts[0, :, fid].cpu().numpy()
        result[fid] = {"mean": float(vals.mean()), "max": float(vals.max()),
                       "active_frac": float((vals > 0).mean())}
    del cache, acts, feat_acts
    torch.cuda.empty_cache()
    return result

## Configuration: Features, Prompts (n=50), Lexicon

In [ ]:
REGISTER_CONFIGS = {
    "legal": {
        "feature_ids": [2294, 12269],
        "multiplier": 8.0,
    },
    "scientific": {
        "feature_ids": [5880],
        "multiplier": 8.0,
    },
    "colloquial": {
        "feature_ids": [1082, 15135],
        "multiplier": 8.0,
    },
    "journalistic": {
        "feature_ids": [16057, 10478],
        "multiplier": 8.0,
    },
}

ALL_FEATURE_IDS = sorted(set(
    fid for cfg in REGISTER_CONFIGS.values() for fid in cfg["feature_ids"]
))

PROMPTS = [
    "O governo anunciou novas medidas para",
    "A universidade publicou um relatório sobre",
    "O Brasil enfrenta desafios na área de",
    "A pesquisa revelou dados importantes sobre",
    "O mercado financeiro reagiu à notícia de",
    "Os moradores reclamaram do problema de",
    "A empresa lançou um novo produto para",
    "O estudo aponta que a população brasileira",
    "A proposta de lei visa regulamentar",
    "O ministro declarou que as políticas de",
    "As mudanças climáticas afetam diretamente",
    "A tecnologia tem transformado a maneira como",
    "O sistema de saúde precisa de melhorias em",
    "A educação no país passa por transformações",
    "Os especialistas alertam para os riscos de",
    "O impacto econômico da pandemia foi",
    "A cultura popular brasileira se manifesta",
    "O desenvolvimento sustentável requer ações",
    "As redes sociais influenciam o comportamento",
    "O transporte público nas grandes cidades",
    "A segurança pública é uma preocupação",
    "O turismo no nordeste brasileiro atrai",
    "A agricultura familiar contribui para",
    "O acesso à justiça no Brasil ainda é",
    "As novas tecnologias de comunicação permitem",
    "O envelhecimento da população traz desafios",
    "A desigualdade social no país se reflete",
    "O esporte brasileiro tem se destacado em",
    "A infraestrutura das cidades precisa de",
    "O consumo de energia renovável cresceu",
    "A inflação nos últimos meses mostrou",
    "O setor industrial enfrentou dificuldades",
    "A reforma tributária propõe mudanças em",
    "O trabalho remoto alterou a dinâmica de",
    "A vacinação contra a gripe é fundamental",
    "O comércio eletrônico expandiu durante",
    "A preservação ambiental depende de",
    "O sistema educacional brasileiro enfrenta",
    "A mobilidade urbana nas capitais requer",
    "O investimento em ciência e tecnologia",
    "A violência urbana tem diminuído em",
    "O patrimônio histórico do país inclui",
    "A alimentação saudável tem ganhado espaço",
    "O mercado de trabalho exige profissionais",
    "A produção literária brasileira contemporânea",
    "O saneamento básico ainda é precário em",
    "A cooperação internacional entre países",
    "O debate sobre inteligência artificial",
    "A inclusão digital no campo avançou com",
    "O papel das instituições democráticas é",
]

print(f"Total de prompts: {len(PROMPTS)}")

LEXICON = {
    "legal": [
        "lei", "decreto", "artigo", "parágrafo", "inciso", "alínea",
        "jurídico", "jurídica", "legislação", "normativo", "normativa",
        "regulamento", "regulamentação", "tribunal", "jurisdição",
        "constitucional", "inconstitucional", "dispositivo", "vigente",
        "revogado", "revogada", "deliberação", "resolução", "portaria",
        "mandato", "mandado", "ministério público", "procuradoria",
        "parecer", "despacho", "sentença", "acórdão", "recurso",
        "impetrante", "requerente", "outorgante", "outorgado",
        "contratante", "contratado", "cláusula", "termo", "aditivo",
        "conforme", "nos termos", "em conformidade", "destarte",
        "outrossim", "doravante", "supracitado", "supracitada",
        "inframencionado", "ad hoc", "ipso facto", "ex officio",
    ],
    "scientific": [
        "hipótese", "metodologia", "resultado", "resultados",
        "análise", "análises", "variável", "variáveis", "amostra",
        "amostras", "estatístico", "estatística", "significativo",
        "significativa", "correlação", "regressão", "coeficiente",
        "desvio padrão", "intervalo de confiança", "p-valor",
        "observou-se", "constatou-se", "verificou-se", "evidencia",
        "evidenciou", "demonstrou", "demonstra", "indicam", "sugere",
        "paradigma", "epistemológico", "ontológico", "fenômeno",
        "fenomenológico", "empírico", "empírica", "teórico", "teórica",
        "framework", "constructo", "operacionalização", "revisão",
        "literatura", "estado da arte", "lacuna", "contribuição",
        "implicações", "limitações", "pesquisadores", "pesquisas",
    ],
    "colloquial": [
        "cara", "mano", "tipo", "né", "tá", "pra", "pro", "num",
        "numa", "dum", "duma", "véi", "velho", "bicho", "massa",
        "show", "top", "foda", "zueira", "zoeira", "zuado", "zoado",
        "galera", "pessoal", "rolê", "rolé", "parada", "bagulho",
        "trampo", "moleque", "mina", "brother", "brô", "da hora",
        "maneiro", "irado", "sinistro", "firmeza", "suave",
        "de boa", "tranquilo", "tlgd", "tmj", "blz", "vlw",
        "kkk", "haha", "hehe", "rs", "rsrs", "kkkkk",
    ],
    "journalistic": [
        "apurou", "apuração", "segundo fontes", "fontes ouvidas",
        "reportagem", "reportagens", "entrevista", "entrevistado",
        "entrevistada", "correspondente", "enviado especial",
        "redação", "pauta", "manchete", "editorial", "colunista",
        "repórter", "jornalista", "exclusivo", "exclusiva",
        "urgente", "última hora", "breaking", "coletiva",
        "coletiva de imprensa", "nota oficial", "comunicado",
        "assessoria", "porta-voz", "informou", "declarou",
        "afirmou", "disse", "apontou", "ressaltou", "destacou",
        "revelou", "denunciou", "confirmou", "negou",
        "segundo o", "de acordo com", "conforme", "ainda segundo",
        "nesta", "neste", "ontem", "hoje", "amanhã",
    ],
}

def count_lexical_markers(text, register):
    text_lower = text.lower()
    count = 0
    for term in LEXICON[register]:
        count += len(re.findall(r'\b' + re.escape(term) + r'\b', text_lower))
    return count

print("Exemplo de contagem:")
test_text = "O decreto nº 1234, conforme artigo 5º, nos termos da lei vigente..."
for reg in LEXICON:
    c = count_lexical_markers(test_text, reg)
    print(f"  {reg}: {c} marcadores")

## Experiment: Generation + Metrics (baseline + 4 registers × 50 prompts)

Each prompt generates 5 texts:
1. Baseline (no steering)
2. Legal (8×)
3. Scientific (8×)
4. Colloquial (8×)
5. Journalistic (8×)

Total: 250 generations + 250 activation measurements

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

results = {
    "prompts": PROMPTS,
    "register_configs": {k: {"feature_ids": v["feature_ids"], "multiplier": v["multiplier"]}
                         for k, v in REGISTER_CONFIGS.items()},
    "per_prompt": [],
}

CHECKPOINT_EVERY = 10
t0 = time.time()

for i, prompt in enumerate(PROMPTS):
    print(f"\n{'='*60}")
    print(f"Prompt {i+1}/{len(PROMPTS)}: {prompt[:50]}...")
    prompt_result = {"prompt": prompt, "baseline": {}, "steered": {}}

    # --- Baseline ---
    print("  Gerando baseline...")
    base_text = generate_baseline(model, tokenizer, prompt,
                                  max_new_tokens=100, temperature=0.7)
    base_acts = measure_feature_activations(model, sae, tokenizer, base_text,
                                            ALL_FEATURE_IDS)
    base_lexical = {reg: count_lexical_markers(base_text, reg) for reg in LEXICON}
    prompt_result["baseline"] = {
        "text": base_text,
        "feature_acts": {str(k): v for k, v in base_acts.items()},
        "lexical_counts": base_lexical,
    }

    # --- Steered (4 registros) ---
    for reg_name, reg_cfg in REGISTER_CONFIGS.items():
        print(f"  Gerando {reg_name} ({reg_cfg['multiplier']}×)...")
        mults = [reg_cfg["multiplier"]] * len(reg_cfg["feature_ids"])
        steered_text = generate_with_steering(
            model, sae, tokenizer, prompt,
            reg_cfg["feature_ids"], mults,
            max_new_tokens=100, temperature=0.7,
        )
        steered_acts = measure_feature_activations(
            model, sae, tokenizer, steered_text, ALL_FEATURE_IDS,
        )
        steered_lexical = {r: count_lexical_markers(steered_text, r) for r in LEXICON}
        prompt_result["steered"][reg_name] = {
            "text": steered_text,
            "feature_acts": {str(k): v for k, v in steered_acts.items()},
            "lexical_counts": steered_lexical,
        }

    results["per_prompt"].append(prompt_result)

    elapsed = time.time() - t0
    per_prompt = elapsed / (i + 1)
    remaining = per_prompt * (len(PROMPTS) - i - 1)
    print(f"  Tempo: {elapsed:.0f}s total | ~{remaining:.0f}s restante")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        ckpt_path = os.path.join(SAVE_DIR, f"register_v2_ckpt_{i+1}.json")
        with open(ckpt_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"  Checkpoint salvo: {ckpt_path}")

total_time = time.time() - t0
print(f"\n{'='*60}")
print(f"Done! Total time: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"Total generations: {len(PROMPTS) * 5}")

## Analysis: Shift Scores + Lexical Counts + Bootstrap CIs

In [ ]:
from scipy.stats import wilcoxon

N_BOOTSTRAP = 10_000
np.random.seed(42)

def bootstrap_ci(data, n_bootstrap=N_BOOTSTRAP, ci=0.95):
    data = np.array(data)
    boot_means = np.array([
        np.mean(np.random.choice(data, size=len(data), replace=True))
        for _ in range(n_bootstrap)
    ])
    alpha = (1 - ci) / 2
    lo = np.percentile(boot_means, alpha * 100)
    hi = np.percentile(boot_means, (1 - alpha) * 100)
    return float(lo), float(hi), float(np.mean(data)), float(np.std(data))

analysis = {}

for reg_name, reg_cfg in REGISTER_CONFIGS.items():
    target_fids = [str(f) for f in reg_cfg["feature_ids"]]

    shift_scores = []
    lexical_diffs = []
    lexical_baseline_counts = []
    lexical_steered_counts = []

    for pr in results["per_prompt"]:
        base_mean = np.mean([
            pr["baseline"]["feature_acts"][fid]["mean"] for fid in target_fids
        ])
        steered_mean = np.mean([
            pr["steered"][reg_name]["feature_acts"][fid]["mean"] for fid in target_fids
        ])
        shift_scores.append(steered_mean - base_mean)

        base_lex = pr["baseline"]["lexical_counts"][reg_name]
        steered_lex = pr["steered"][reg_name]["lexical_counts"][reg_name]
        lexical_diffs.append(steered_lex - base_lex)
        lexical_baseline_counts.append(base_lex)
        lexical_steered_counts.append(steered_lex)

    shift_lo, shift_hi, shift_mean, shift_std = bootstrap_ci(shift_scores)
    lex_lo, lex_hi, lex_mean, lex_std = bootstrap_ci(lexical_diffs)

    try:
        stat_shift, p_shift = wilcoxon(shift_scores, alternative='greater')
    except ValueError:
        stat_shift, p_shift = float('nan'), 1.0

    try:
        stat_lex, p_lex = wilcoxon(lexical_diffs, alternative='greater')
    except ValueError:
        stat_lex, p_lex = float('nan'), 1.0

    cross_lex = {}
    for other_reg in LEXICON:
        other_diffs = []
        for pr in results["per_prompt"]:
            b = pr["baseline"]["lexical_counts"][other_reg]
            s = pr["steered"][reg_name]["lexical_counts"][other_reg]
            other_diffs.append(s - b)
        clo, chi, cmean, cstd = bootstrap_ci(other_diffs)
        cross_lex[other_reg] = {
            "mean_diff": cmean, "std": cstd,
            "ci_95": [clo, chi],
        }

    analysis[reg_name] = {
        "shift_score": {
            "mean": shift_mean, "std": shift_std,
            "ci_95": [shift_lo, shift_hi],
            "wilcoxon_stat": float(stat_shift), "wilcoxon_p": float(p_shift),
            "n": len(shift_scores),
            "values": shift_scores,
        },
        "lexical_diff": {
            "target_register": reg_name,
            "mean": lex_mean, "std": lex_std,
            "ci_95": [lex_lo, lex_hi],
            "wilcoxon_stat": float(stat_lex), "wilcoxon_p": float(p_lex),
            "baseline_total": sum(lexical_baseline_counts),
            "steered_total": sum(lexical_steered_counts),
            "mean_baseline": float(np.mean(lexical_baseline_counts)),
            "mean_steered": float(np.mean(lexical_steered_counts)),
            "n": len(lexical_diffs),
            "values": lexical_diffs,
        },
        "cross_lexical": cross_lex,
    }

print("\n" + "=" * 80)
print("RESULTADOS — REGISTER STEERING v2 (n=50 prompts)")
print("=" * 80)

print(f"\n{'Register':<14} {'Shift Score':>14} {'95% CI':>22} {'p-value':>10}")
print("-" * 65)
for reg_name in REGISTER_CONFIGS:
    a = analysis[reg_name]["shift_score"]
    sig = "***" if a["wilcoxon_p"] < 0.001 else "**" if a["wilcoxon_p"] < 0.01 else "*" if a["wilcoxon_p"] < 0.05 else ""
    print(f"{reg_name:<14} {a['mean']:>+.4f} ± {a['std']:.4f}  [{a['ci_95'][0]:>+.4f}, {a['ci_95'][1]:>+.4f}]  {a['wilcoxon_p']:.4f} {sig}")

print(f"\n{'Register':<14} {'Lex Δ (target)':>14} {'95% CI':>22} {'p-value':>10} {'base→steered':>16}")
print("-" * 80)
for reg_name in REGISTER_CONFIGS:
    a = analysis[reg_name]["lexical_diff"]
    sig = "***" if a["wilcoxon_p"] < 0.001 else "**" if a["wilcoxon_p"] < 0.01 else "*" if a["wilcoxon_p"] < 0.05 else ""
    print(f"{reg_name:<14} {a['mean']:>+.2f} ± {a['std']:.2f}      [{a['ci_95'][0]:>+.2f}, {a['ci_95'][1]:>+.2f}]  {a['wilcoxon_p']:.4f} {sig}   {a['mean_baseline']:.1f}→{a['mean_steered']:.1f}")

print("\nCross-lexical (marcadores de OUTROS registros no texto steered):")
print(f"{'Steered→':<14}", end="")
for r in LEXICON:
    print(f"  {r:>12}", end="")
print()
for reg_name in REGISTER_CONFIGS:
    print(f"{reg_name:<14}", end="")
    for other in LEXICON:
        val = analysis[reg_name]["cross_lexical"][other]["mean_diff"]
        print(f"  {val:>+12.2f}", end="")
    print()

## Save final results

In [ ]:
final_results = {
    "experiment": "register_steering_v2",
    "n_prompts": len(PROMPTS),
    "n_bootstrap": N_BOOTSTRAP,
    "registers": list(REGISTER_CONFIGS.keys()),
    "analysis": {},
}

for reg_name in REGISTER_CONFIGS:
    a = analysis[reg_name]
    final_results["analysis"][reg_name] = {
        "shift_score": {
            "mean": a["shift_score"]["mean"],
            "std": a["shift_score"]["std"],
            "ci_95": a["shift_score"]["ci_95"],
            "wilcoxon_p": a["shift_score"]["wilcoxon_p"],
            "n": a["shift_score"]["n"],
        },
        "lexical_diff": {
            "mean": a["lexical_diff"]["mean"],
            "std": a["lexical_diff"]["std"],
            "ci_95": a["lexical_diff"]["ci_95"],
            "wilcoxon_p": a["lexical_diff"]["wilcoxon_p"],
            "mean_baseline": a["lexical_diff"]["mean_baseline"],
            "mean_steered": a["lexical_diff"]["mean_steered"],
        },
        "cross_lexical": a["cross_lexical"],
    }

output_path = os.path.join(SAVE_DIR, "exp_register_steering_v2_results.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)
print(f"Resultados salvos: {output_path}")

full_path = os.path.join(SAVE_DIR, "exp_register_steering_v2_full.json")
with open(full_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Dados completos salvos: {full_path}")

print("\nResumo para o paper:")
print("=" * 60)
for reg_name in REGISTER_CONFIGS:
    a = analysis[reg_name]
    ss = a["shift_score"]
    lex = a["lexical_diff"]
    sig_ss = "p < 0.001" if ss["wilcoxon_p"] < 0.001 else f"p = {ss['wilcoxon_p']:.3f}"
    sig_lex = "p < 0.001" if lex["wilcoxon_p"] < 0.001 else f"p = {lex['wilcoxon_p']:.3f}"
    print(f"{reg_name}:")
    print(f"  Shift score: {ss['mean']:+.4f} [{ss['ci_95'][0]:+.4f}, {ss['ci_95'][1]:+.4f}], {sig_ss}")
    print(f"  Lexical Δ:   {lex['mean']:+.2f} [{lex['ci_95'][0]:+.2f}, {lex['ci_95'][1]:+.2f}], {sig_lex}")
    print(f"  Baseline→Steered: {lex['mean_baseline']:.1f}→{lex['mean_steered']:.1f} marcadores/prompt")
    print()

## Feature Discovery: Which features does steering activate beyond the known ones?

For each generated text (250 texts = 50 baselines + 4 registers × 50), we encode with the SAE and measure ALL 16k features. We then identify:
1. Features with the highest **inter-register variance** (the most discriminative)
2. Features that activate **exclusively** in one register
3. New features not in our original set of 7

In [ ]:
def get_all_feature_acts(model, sae, tokenizer, text, hook_name=HOOK_NAME):
    input_ids = tokenizer.encode(text, return_tensors="pt").to(device)
    if input_ids.shape[1] > 512:
        input_ids = input_ids[:, :512]
    with torch.no_grad():
        _, cache = model.run_with_cache(input_ids, names_filter=lambda n: n == hook_name)
        resid = cache[hook_name]
        feat_acts = sae.encode(resid)
        mean_acts = feat_acts[0].mean(dim=0).cpu()
    del cache, resid, feat_acts
    torch.cuda.empty_cache()
    return mean_acts

conditions = ["baseline", "legal", "scientific", "colloquial", "journalistic"]
N_FEATURES = sae.cfg.d_sae
n_prompts = len(results["per_prompt"])

all_acts = {cond: torch.zeros(n_prompts, N_FEATURES) for cond in conditions}

print(f"Encodando {n_prompts * len(conditions)} textos...")
t0 = time.time()

for i, pr in enumerate(results["per_prompt"]):
    if (i + 1) % 10 == 0 or i == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (n_prompts - i - 1)
        print(f"  Prompt {i+1}/{n_prompts} | {elapsed:.0f}s elapsed | ~{eta:.0f}s remaining")

    all_acts["baseline"][i] = get_all_feature_acts(
        model, sae, tokenizer, pr["baseline"]["text"]
    )
    for reg in ["legal", "scientific", "colloquial", "journalistic"]:
        all_acts[reg][i] = get_all_feature_acts(
            model, sae, tokenizer, pr["steered"][reg]["text"]
        )

print(f"Done in {time.time() - t0:.0f}s")
print(f"Shape of each condition: {all_acts['baseline'].shape}")

In [ ]:
KNOWN_FEATURES = set(ALL_FEATURE_IDS)

cond_means = {}
for cond in conditions:
    cond_means[cond] = all_acts[cond].mean(dim=0).numpy()

stacked = np.stack([cond_means[c] for c in conditions], axis=0)
inter_register_var = np.var(stacked, axis=0)

steered_only = np.stack([cond_means[c] for c in conditions if c != "baseline"], axis=0)
inter_steered_var = np.var(steered_only, axis=0)

active_mask = np.max(stacked, axis=0) > 0.01
inter_register_var_active = inter_register_var.copy()
inter_register_var_active[~active_mask] = 0

ranked = np.argsort(-inter_register_var_active)

print("=" * 90)
print("TOP 30 FEATURES WITH HIGHEST INTER-REGISTER VARIANCE")
print("(mean activation per condition — includes baseline)")
print("=" * 90)
print(f"{'Rank':<5} {'FeatID':<8} {'Var':>8} {'baseline':>10} {'legal':>10} {'scientific':>10} {'colloquial':>10} {'journal.':>10} {'Known?':>7}")
print("-" * 90)

top_new = []
for rank, fid in enumerate(ranked[:30]):
    is_known = "✓" if fid in KNOWN_FEATURES else ""
    vals = {c: cond_means[c][fid] for c in conditions}
    peak_cond = max(conditions, key=lambda c: vals[c])
    print(f"{rank+1:<5} {fid:<8} {inter_register_var_active[fid]:>8.4f} "
          f"{vals['baseline']:>10.4f} {vals['legal']:>10.4f} "
          f"{vals['scientific']:>10.4f} {vals['colloquial']:>10.4f} "
          f"{vals['journalistic']:>10.4f} {is_known:>7}")
    if fid not in KNOWN_FEATURES:
        top_new.append((fid, inter_register_var_active[fid], peak_cond))

print(f"\nDas top 30, {len(top_new)} are NEW (not in the original set)")
print(f"Features conhecidas no set original: {KNOWN_FEATURES}")

In [ ]:
from scipy.stats import kruskal

print("=" * 90)
print("SPECIFICITY ANALYSIS: Features with significantly different activation across registers")
print("(Kruskal-Wallis over 50 prompts × 5 conditions)")
print("=" * 90)

kw_results = []

for fid in range(N_FEATURES):
    groups = [all_acts[c][:, fid].numpy() for c in conditions]
    if all(np.max(g) < 0.01 for g in groups):
        continue
    try:
        stat, p = kruskal(*groups)
        if p < 0.01:
            peak_cond = conditions[np.argmax([np.mean(g) for g in groups])]
            effect = inter_register_var_active[fid]
            kw_results.append((fid, stat, p, peak_cond, effect))
    except ValueError:
        pass

kw_results.sort(key=lambda x: -x[1])

print(f"\nFeatures com p < 0.01 no Kruskal-Wallis: {len(kw_results)}")

print(f"\n{'Rank':<5} {'FeatID':<8} {'H-stat':>8} {'p-value':>12} {'Peak':>14} {'Known?':>7}")
print("-" * 60)
for rank, (fid, stat, p, peak, eff) in enumerate(kw_results[:40]):
    is_known = "✓" if fid in KNOWN_FEATURES else ""
    p_str = f"{p:.2e}" if p < 0.001 else f"{p:.4f}"
    print(f"{rank+1:<5} {fid:<8} {stat:>8.1f} {p_str:>12} {peak:>14} {is_known:>7}")

known_in_top = sum(1 for fid, *_ in kw_results[:40] if fid in KNOWN_FEATURES)
print(f"\nOf the top 40 KW, {known_in_top} are known features from the original set")

new_discoveries = [(fid, stat, p, peak) for fid, stat, p, peak, _ in kw_results
                   if fid not in KNOWN_FEATURES]
print(f"Total de features NOVAS com p < 0.01: {len(new_discoveries)}")

by_peak = {}
for fid, stat, p, peak in new_discoveries[:100]:
    by_peak.setdefault(peak, []).append(fid)

print("\nDistribution of the top 100 new features by preferred register:")
for cond in conditions:
    fids = by_peak.get(cond, [])
    print(f"  {cond:<14}: {len(fids)} features — ex: {fids[:5]}")

In [ ]:
print("=" * 90)
print("STEERING LEAKAGE: O steering de um registro ativa features de OUTROS registros?")
print("=" * 90)

for reg in ["legal", "scientific", "colloquial", "journalistic"]:
    diffs = all_acts[reg] - all_acts["baseline"]
    mean_diff = diffs.mean(dim=0).numpy()

    top_up = np.argsort(-mean_diff)[:20]
    cfg_fids = set(REGISTER_CONFIGS[reg]["feature_ids"])

    print(f"\n--- Steering: {reg} (features alvo: {cfg_fids}) ---")
    print(f"{'Rank':<5} {'FeatID':<8} {'Δ mean':>10} {'Is Target':>10} {'Known Register':>16}")
    print("-" * 55)

    for rank, fid in enumerate(top_up):
        is_target = "TARGET" if fid in cfg_fids else ""
        known_reg = ""
        for r, c in REGISTER_CONFIGS.items():
            if fid in c["feature_ids"]:
                known_reg = r
                break
        print(f"{rank+1:<5} {fid:<8} {mean_diff[fid]:>+10.4f} {is_target:>10} {known_reg:>16}")

print("\n\n" + "=" * 90)
print("CO-ACTIVATION: When steering legal, do colloquial features decrease?")
print("=" * 90)

for reg in ["legal", "scientific", "colloquial", "journalistic"]:
    print(f"\n--- Steering: {reg} ---")
    for other_reg, other_cfg in REGISTER_CONFIGS.items():
        diffs_other = []
        for fid in other_cfg["feature_ids"]:
            d = (all_acts[reg][:, fid] - all_acts["baseline"][:, fid]).mean().item()
            diffs_other.append(d)
        mean_d = np.mean(diffs_other)
        direction = "↑" if mean_d > 0.001 else "↓" if mean_d < -0.001 else "≈"
        print(f"  Features de {other_reg:<14}: Δ = {mean_d:>+.4f} {direction}")

In [ ]:
discovery_results = {
    "kruskal_wallis": {
        "total_significant_p01": len(kw_results),
        "total_new_features": len(new_discoveries),
        "top_40": [
            {"feature_id": int(fid), "H_stat": float(stat),
             "p_value": float(p), "peak_condition": peak,
             "is_known": fid in KNOWN_FEATURES}
            for fid, stat, p, peak, _ in kw_results[:40]
        ],
    },
    "variance_ranking": {
        "top_30": [
            {"feature_id": int(ranked[i]),
             "variance": float(inter_register_var_active[ranked[i]]),
             "per_condition": {c: float(cond_means[c][ranked[i]]) for c in conditions},
             "is_known": int(ranked[i]) in KNOWN_FEATURES}
            for i in range(30)
        ],
    },
    "steering_leakage": {},
    "coactivation": {},
}

for reg in ["legal", "scientific", "colloquial", "journalistic"]:
    diffs = all_acts[reg] - all_acts["baseline"]
    mean_diff = diffs.mean(dim=0).numpy()
    top_up = np.argsort(-mean_diff)[:20]
    discovery_results["steering_leakage"][reg] = [
        {"feature_id": int(fid), "mean_diff": float(mean_diff[fid]),
         "is_target": int(fid) in set(REGISTER_CONFIGS[reg]["feature_ids"])}
        for fid in top_up
    ]

    coact = {}
    for other_reg, other_cfg in REGISTER_CONFIGS.items():
        diffs_other = []
        for fid in other_cfg["feature_ids"]:
            d = (all_acts[reg][:, fid] - all_acts["baseline"][:, fid]).mean().item()
            diffs_other.append(d)
        coact[other_reg] = float(np.mean(diffs_other))
    discovery_results["coactivation"][reg] = coact

disc_path = os.path.join(SAVE_DIR, "exp_register_discovery.json")
with open(disc_path, "w", encoding="utf-8") as f:
    json.dump(discovery_results, f, ensure_ascii=False, indent=2)
print(f"Discovery results salvos: {disc_path}")

print("\nResumo:")
print(f"  Features with significant inter-register variance (KW p<0.01): {len(kw_results)}")
print(f"  Dessas, NOVAS (fora do set original): {len(new_discoveries)}")
print(f"  Features conhecidas validadas: {len(kw_results) - len(new_discoveries)}")